In [1]:
import os
import sys

sys.path.append(os.path.abspath(".."))  # raíz de backend/
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "config.settings")
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

import django
django.setup()

In [2]:
# Buscamos los archivos.

from pathlib import Path
from django.conf import settings
import pandas as pd

RAW = Path(settings.BASE_DIR).parent / "datasets" / "raw"
archivos = sorted(RAW.glob("BCCh Gasto PIB*.xlsx"))
[a.name for a in archivos]

['BCCh Gasto PIB nominal anual.xlsx', 'BCCh Gasto PIB real anual.xlsx']

In [ ]:
# Hacemos las funciones de lectura.

""" Lee la hoja “Cuadro” con los valores y la hoja “Series” con los códigos.
Une ambas por posición, porque las filas vienen en el mismo orden. Si la cantidad de filas no coincide, lanza un error en vez de mezclar datos.
Calcula el nivel de la jerarquía a partir de la sangría del nombre: cada nivel son 3 espacios.
Pasa la tabla a formato largo, con una fila por año y serie, que es el formato que tendrá en la base de datos. """


def leer_gasto_pib(ruta):
    cuadro = pd.read_excel(ruta, sheet_name="Cuadro", header=2)
    series = pd.read_excel(ruta, sheet_name="Series")

    cuadro = cuadro.dropna(subset=["Reg"]).reset_index(drop=True)
    if len(cuadro) != len(series):
        raise ValueError(f"{ruta.name}: {len(cuadro)} filas en Cuadro vs {len(series)} en Series")

    cuadro.insert(0, "codigo", series["Serie"].values)
    cuadro.insert(1, "descripcion_serie", series["Descripción"].values)

    nombre = cuadro["Descripción series"].astype(str).str.replace("\xa0", " ")
    cuadro["nivel"] = (nombre.str.len() - nombre.str.lstrip().str.len()) // 3
    cuadro["nombre"] = nombre.str.strip()
    cuadro["orden"] = range(len(cuadro))

    columnas_anio = [c for c in cuadro.columns if hasattr(c, "year")]
    largo = cuadro.melt(
        id_vars=["codigo", "descripcion_serie", "nombre", "nivel", "orden"],
        value_vars=columnas_anio,
        var_name="fecha",
        value_name="valor",
    )
    largo["anio"] = pd.to_datetime(largo["fecha"]).dt.year
    largo["medida"] = largo["codigo"].str.split(".").str[3].map({"N": "nominal", "R": "real"})
    return largo.drop(columns="fecha")


df = pd.concat([leer_gasto_pib(a) for a in archivos], ignore_index=True)
df.shape

(1590, 8)

In [4]:
# Revisamos que la lectura es correcta y no haya desface entre las filas y columnas de ambos documentos.

pd.set_option("display.max_colwidth", 60)
(df.drop_duplicates(["medida", "codigo"])
   [["medida", "orden", "nivel", "nombre", "descripcion_serie"]])

,medida,orden,nivel,nombre,descripcion_serie
0,nominal,0,0,Demanda Interna,"Demanda Interna a precios corrientes serie empalmada, re..."
1,nominal,1,0,Consumo total,"Consumo total a precios corrientes, series empalmadas, r..."
2,nominal,2,1,Consumo de hogares e IPSFL,"Consumo privado a precios corrientes serie empalmada, re..."
3,nominal,3,2,Bienes durables,"Consumo bienes durables, a precios corrientes serie empa..."
4,nominal,4,2,Bienes no durables,"Consumo bienes no durables, a precios corrientes serie e..."
5,nominal,5,2,Servicios,"Consumo servicios, a precios corrientes serie empalmada,..."
6,nominal,6,1,Consumo Gobierno,"Consumo gobierno a precios corrientes serie empalmada, r..."
7,nominal,7,0,Formación bruta de capital,"Formación bruta de capital a precios corrientes, series ..."
8,nominal,8,0,Formación bruta de capital fijo,"Formación bruta capital fijo a precios corrientes, serie..."
9,nominal,9,1,Construcción y otras obras,"Construcción y otras obras de inversión, a precios corri..."


In [5]:
# Revisando la cobertura y los espacios vacíos del documento, el original tiene algunos pero no debeira haber nuevos.

print(df.groupby("medida")["anio"].agg(["min", "max", "count"]))
print()
print("Vacíos:")
print(df[df["valor"].isna()].groupby(["medida", "nombre"])["anio"].agg(["min", "max", "count"]))
print()
print("Duplicados año+código:", df.duplicated(["anio", "codigo"]).sum())

          min   max  count
medida                    
nominal  1996  2025    810
real     1996  2025    780

Vacíos:
                 min   max  count
medida  nombre                   
nominal Cobre   1996  2012     17
        Resto   1996  2012     17
real    Cobre   1996  2012     17
        Resto   1996  2012     17

Duplicados año+código: 0


In [6]:
# Identidad contable.

nom = df[df["medida"] == "nominal"].pivot(index="anio", columns="codigo", values="valor")
c = lambda clave: nom[[col for col in nom.columns if f".{clave}." in col][0]]

diferencia = c("DDI") + c("XBS") - c("IBS") - c("PIB")
print("Máxima diferencia DI + X − M − PIB:", diferencia.abs().max())

Máxima diferencia DI + X − M − PIB: 2.9976945370435715e-09


In [7]:
# Primer vistazo al dataframe que contiene ambas tablas.

real = df[df["medida"] == "real"].pivot(index="anio", columns="codigo", values="valor")
r = lambda clave: real[[col for col in real.columns if f".{clave}." in col][0]]

resumen = pd.DataFrame({
    "PIB nominal (billones)": (c("PIB") / 1000).round(1),
    "Crecimiento real %": (r("PIB").pct_change() * 100).round(2),
    "Consumo gobierno % PIB": (c("COG") / c("PIB") * 100).round(1),
    "Importaciones % PIB": (c("IBS") / c("PIB") * 100).round(1),
})
resumen

,PIB nominal (billones),Crecimiento real %,Consumo gobierno % PIB,Importaciones % PIB
anio,,,,
1996,32.4,NaN,11.3,28.0
1997,35.9,7.39,11.4,28.2
1998,37.7,4.18,11.8,28.7
1999,38.5,-0.27,12.7,26.4
2000,42.2,4.97,12.6,28.6
2001,45.4,3.15,12.6,30.3
2002,48.4,3.20,12.7,30.2
2003,52.9,4.72,12.2,30.6
2004,60.4,6.67,11.4,30.2


**La previsualización completa ahora empieza la ingestión.**

In [3]:
import logging
logging.basicConfig(level=logging.INFO)

from data_engine.services.ingestion.cuenta_nacional_gasto import ingerir
ingerir()

INFO:data_engine.services.ingestion.cuenta_nacional_gasto:Archivos: ['BCCh Gasto PIB nominal anual.xlsx', 'BCCh Gasto PIB real anual.xlsx']
INFO:data_engine.services.ingestion.cuenta_nacional_gasto:Gasto del PIB (BCCh): {'creados': 0, 'actualizados': 0, 'sin_cambios': 1590}


{'creados': 0, 'actualizados': 0, 'sin_cambios': 1590}

In [4]:
from data_engine.models import CuentaNacionalGasto
from data_engine.models.cuenta_nacional_gasto import CLAVE_PIB, MedidaCuentaNacional

list(CuentaNacionalGasto.objects.serie(CLAVE_PIB, MedidaCuentaNacional.NOMINAL))[-2:]

[{'anio': 2024, 'valor': Decimal('310681.169647')},
 {'anio': 2025, 'valor': Decimal('339977.710114')}]